In [ ]:
import requests
from bs4 import BeautifulSoup
import re
import time
import random
import pandas as pd

In [ ]:
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
    "Accept-Language": "vi-VN,vi;q=0.9",
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,image/avif,image/webp,*/*;q=0.8"
}

base_url = "https://bonbanh.com"
all_cars_data = []

# Dùng Session để lưu Cookie, tránh bị Bonbanh chặn bot
session = requests.Session()
session.headers.update(headers)
print("Đang khởi tạo Session & vượt mồi Cookie...")
session.get(base_url, timeout=10)

start_page = 1
end_page = 5 # Test thử 5 trang trước

print("\n🚀 Bắt đầu thu thập dữ liệu xe...")

for page in range(start_page, end_page + 1):
    list_url = f"{base_url}/oto/page,{page}"
    print(f"\n[+] Đang quét trang {page}...")
    
    try:
        res = session.get(list_url, timeout=10)
        soup = BeautifulSoup(res.text, "html.parser")
        
        # SỬ DỤNG CSS SELECTOR: Lấy tất cả các thẻ có chứa class 'car-item' (rất chuẩn xác)
        car_items = soup.select(".car-item")
        
        if not car_items:
            print(f"  [!] Không tìm thấy xe ở trang {page}. Có thể bị chặn IP, hãy thử chờ 1 lúc.")
            continue
            
        for item in car_items:
            # Tìm thẻ a đầu tiên chứa link
            a_tag = item.select_one("a[href]")
            if not a_tag: continue
            
            car_link = base_url + "/" + a_tag['href']
            
            try:
                print(f"  -> Bóc tách: {car_link}")
                detail_res = session.get(car_link, timeout=15)
                detail_soup = BeautifulSoup(detail_res.text, "html.parser")
                
                car_info = {"URL": car_link}
                
                # --- 1. Lấy Mã tin bằng Regex dựa trên hình ảnh của bạn ---
                # Cú pháp này sẽ tìm đoạn "Mã tin : 6690962" ở bất cứ đâu trong text của trang
                code_match = re.search(r'Mã tin\s*:\s*(\d+)', detail_soup.text)
                if code_match:
                    car_info["Mã_Tin"] = code_match.group(1)
                
                # --- 2. Lấy Hãng và Dòng xe từ thanh điều hướng (Breadcrumb) ---
                # Tìm tất cả thẻ <a> nằm trong div có id='breadcrum' hoặc 'breadcrumb'
                breadcrumb_links = detail_soup.select("#breadcrum a, #breadcrumb a")
                if len(breadcrumb_links) >= 4:
                    car_info["Hãng_Xe"] = breadcrumb_links[2].text.strip()
                    car_info["Dòng_Xe"] = breadcrumb_links[3].text.strip()
                
                # --- 3. Lấy Tên xe & Giá (Thẻ h1) ---
                title_tag = detail_soup.select_one("h1")
                if title_tag:
                    full_title = title_tag.text.strip()
                    if " - " in full_title:
                        car_info["Tên_Xe"] = full_title.rsplit(" - ", 1)[0].replace("Xe ", "").strip()
                        car_info["Giá_Tiền"] = full_title.rsplit(" - ", 1)[1].strip()
                    else:
                        car_info["Tên_Xe"] = full_title
                
                # --- 4. Lấy bảng Thông số kỹ thuật (Cực kỳ quan trọng cho Data Science) ---
                spec_rows = detail_soup.select(".row, .row_last")
                for row in spec_rows:
                    lbl = row.select_one("label")
                    inp = row.select_one(".inp")
                    if lbl and inp:
                        key = lbl.text.replace(":", "").strip()
                        val = inp.text.strip()
                        car_info[key] = val
                        
                # --- 5. Lấy Thông tin mô tả ---
                des_tag = detail_soup.select_one(".des_txt, .des-txt")
                if des_tag:
                    car_info["Mô_Tả"] = des_tag.text.strip().replace("\n", " | ")
                    
                # --- 6. Lấy Thông tin liên hệ ---
                contact_name = detail_soup.select_one(".contact-txt .cname, .contact_txt .cname")
                if contact_name:
                    car_info["Người_Liên_Hệ"] = contact_name.text.strip()
                
                # Đưa vào list tổng
                all_cars_data.append(car_info)
                
                # Trễ ngẫu nhiên 1 - 2.5 giây để đóng giả người thật xem web
                time.sleep(random.uniform(1, 2.5))
                
            except Exception as e:
                print(f"     [!] Bỏ qua xe này vì lỗi bóc tách: {e}")
                continue
                
    except Exception as e:
        print(f"[!] Lỗi kết nối trang {page}: {e}")
        continue

# Xuất dữ liệu
df = pd.DataFrame(all_cars_data)
df.to_csv('raw_data_bonbanh_v2.csv', index=False, encoding='utf-8-sig')

print(f"\n✅ THÀNH CÔNG! Đã thu thập trọn vẹn {len(all_cars_data)} chiếc xe.")

In [ ]:
import pandas as pd
import re

# 1. ĐỌC DỮ LIỆU THÔ (Raw Data)
df = pd.read_csv('raw_data_bonbanh_v2.csv')

# Hàm làm sạch và tách Hãng, Tên, Giá
def clean_car_name(raw_name):
    # Loại bỏ các ký tự ẩn \n, \t và thay bằng 1 khoảng trắng
    clean_str = re.sub(r'\s+', ' ', str(raw_name)).strip()
    
    # Xóa chữ "Xe " vô nghĩa ở đầu câu
    if clean_str.startswith("Xe "):
        clean_str = clean_str[3:]
        
    # Tách Tên Xe và Giá Tiền dựa vào dấu " - "
    if " - " in clean_str:
        ten_xe, gia_chu = clean_str.rsplit(" - ", 1)
    else:
        ten_xe, gia_chu = clean_str, "Thỏa thuận"
        
    return pd.Series([ten_xe.strip(), gia_chu.strip()])

# Áp dụng hàm làm sạch vào cột Tên_Xe
df[['Tên_Xe_Chuẩn', 'Giá_Bằng_Chữ']] = df['Tên_Xe'].apply(clean_car_name)

# 2. CHUẨN HÓA GIÁ TIỀN THÀNH SỐ (Triệu VNĐ)
def convert_price_to_number(price_str):
    if not isinstance(price_str, str) or 'thỏa thuận' in price_str.lower(): 
        return None
    
    price_str = price_str.lower()
    total_trieu = 0
    
    # Bóc tách số Tỷ (1 Tỷ = 1000 Triệu)
    ty_match = re.search(r'(\d+)\s*tỷ', price_str)
    if ty_match:
        total_trieu += int(ty_match.group(1)) * 1000
        
    # Bóc tách số Triệu
    trieu_match = re.search(r'(\d+)\s*triệu', price_str)
    if trieu_match:
        total_trieu += int(trieu_match.group(1))
        
    return total_trieu if total_trieu > 0 else None

df['Giá_Triệu_VNĐ'] = df['Giá_Bằng_Chữ'].apply(convert_price_to_number)

# 3. CHUẨN HÓA SỐ KM (Odo) THÀNH SỐ NGUYÊN
def convert_odo(odo_str):
    if not isinstance(odo_str, str): return None
    nums = re.findall(r'\d+', odo_str)
    if not nums: return None
    return int(''.join(nums))

df['Odo_Km'] = df['Số Km đã đi'].apply(convert_odo)

# 4. TÁCH HÃNG XE TỪ TÊN XE CHUẨN
# Danh sách các hãng phổ biến tại VN
brands = ["Mercedes Benz", "LandRover", "Rolls Royce", "Audi", "BMW", "Bentley", 
          "Ford", "Honda", "Hyundai", "Kia", "Lexus", "Mazda", "MG", "Mitsubishi", 
          "Nissan", "Peugeot", "Porsche", "Subaru", "Suzuki", "Toyota", "VinFast"]

def extract_brand(name):
    for b in brands:
        if name.lower().startswith(b.lower()):
            return b
    return "Khác"

df['Hãng_Xe'] = df['Tên_Xe_Chuẩn'].apply(extract_brand)

# 5. DỌN DẸP LẠI BẢNG VÀ XUẤT RA CLEAN DATA
# Xóa bỏ các cột thô gồ ghề ban đầu
columns_to_drop = ['Tên_Xe', 'Giá_Bằng_Chữ', 'Số Km đã đi']
df_clean = df.drop(columns=columns_to_drop)

# Đẩy các cột quan trọng lên đầu cho đẹp
cols = ['Mã_Tin', 'Hãng_Xe', 'Tên_Xe_Chuẩn', 'Năm sản xuất', 'Giá_Triệu_VNĐ', 'Odo_Km', 'Màu ngoại thất'] + \
       [c for c in df_clean.columns if c not in ['Mã_Tin', 'Hãng_Xe', 'Tên_Xe_Chuẩn', 'Năm sản xuất', 'Giá_Triệu_VNĐ', 'Odo_Km', 'Màu ngoại thất']]
df_clean = df_clean[cols]

# Xuất file clean_data.csv để nộp cho Giảng viên
df_clean.to_csv('clean_data.csv', index=False, encoding='utf-8-sig')

print("🎉 Dữ liệu đã được làm sạch và lưu thành clean_data.csv")
df_clean.head(5) # In ra 5 dòng đầu tiên xem thử